# Real-World Dataset Analysis Project

**Internship:** YuvaIntern | **Project:** Real-World Dataset Analysis Project | **Student:** Lokesh Patwal, B.Tech (Invertis University)

## 1. Project Title and Overview

This notebook analyses a real-world retail dataset, the **Global Superstore Sales dataset**, from start to finish:

1. load the data and understand its structure,
2. check data quality and clean the data (without changing the original),
3. explore it with Pandas, NumPy and statistics (EDA),
4. draw charts with Matplotlib and Seaborn,
5. write findings, business recommendations, limitations and a conclusion.

> **How to use this notebook:** download the dataset (see Section 4 and `data/README.md`), save it in the `data` folder, then choose **Kernel > Restart & Run All**. Every number, table and interpretation below is calculated from *your* data file when you run the notebook, so the results are never typed in by hand.

## 2. Problem Statement

A global retailer sells products in many countries and markets. Management needs to know **where money is made and where it is lost**. The problem solved in this project is:

> *Using the order data, which categories, sub-categories, products, markets and customers generate the most sales and profit, how do sales change over time, and how do discounts affect profitability?*

The answers should lead to practical, evidence-based recommendations.

## 3. Objectives

1. Select a reliable real-world dataset and understand its business context.
2. Explore the dataset structure, data types, distributions, missing values, duplicates and relationships.
3. Clean and preprocess the data appropriately (and document every change).
4. Perform Exploratory Data Analysis (EDA) using Pandas, NumPy and statistical techniques.
5. Create meaningful visualizations using Matplotlib and Seaborn.
6. Present findings, interpretations, conclusions and recommendations professionally.

## 4. Dataset Source and Description

* **Dataset:** Global Superstore (file `Global_Superstore2.csv`; an Excel version `Global_Superstore2.xlsx` is in the same download)
* **Source:** Kaggle, dataset `apoorvaappz/global-super-store-dataset`. Page: https://www.kaggle.com/datasets/apoorvaappz/global-super-store-dataset
* **Size (as reported by published notebooks that use this file):** 51,290 rows x 24 columns. The notebook checks the real size when it loads the file.
* **License:** I could not verify the license from outside Kaggle. Check the "License" field on the Kaggle page. Because it is unclear, the data file is **not** included in this project's GitHub repository.
* **Download:** see `data/README.md` (manual download and Kaggle command-line options).

**Columns (names as in the file; meanings follow the public dataset descriptions, please verify on the Kaggle page):**

| Column | Meaning |
|---|---|
| Row ID | Unique number of each row |
| Order ID | Identifier of the order (one order can have several lines) |
| Order Date / Ship Date | Date the order was placed / shipped |
| Ship Mode | Delivery method (e.g. Same Day, First Class, Second Class, Standard Class) |
| Customer ID / Customer Name | Customer identifier / name |
| Segment | Customer segment (e.g. Consumer, Corporate, Home Office) |
| City / State / Country / Postal Code | Delivery location |
| Market / Region | Broad geographic market and region |
| Product ID / Category / Sub-Category / Product Name | Product identifier and product hierarchy |
| Sales | Sales amount of the order line |
| Quantity | Number of items in the order line |
| Discount | Discount rate applied (0 to 1; 0.2 = 20%) |
| Profit | Profit of the order line (negative = loss) |
| Shipping Cost | Cost of shipping the order line |
| Order Priority | Priority of the order (e.g. Critical, High, Medium, Low) |

The currency is not stated in the documentation I could verify, so amounts are shown as plain numbers.

## 5. Business Context

The dataset describes order lines of a global retailer selling products in several categories to customers in many countries. In such a business the main questions are:

* Which categories and products earn the most profit, and which lose money?
* Which markets are strong or weak?
* Are there busy and quiet seasons?
* Do discounts increase sales at the cost of profit?
* Which customers and segments matter most?

These are **research questions**, not conclusions. The notebook answers them with the actual data.

## 6. Python Environment and Libraries

* **Python** + **Jupyter Notebook**: the working environment
* **Pandas**: tables (DataFrames), cleaning, grouping
* **NumPy**: numerical calculations
* **Matplotlib / Seaborn**: charts
* **SciPy**: statistical tests (correlation, Kruskal-Wallis, Mann-Whitney)

The next cell imports the libraries, makes the `src/analysis.py` helper functions available, and holds the **configuration**. If your file is not in the `data` folder, change `DATA_PATH`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

# --- Find the project folder (the notebook lives in the project's root folder) ---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():          # notebook started from another folder?
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import analysis as ra                            # helper functions written for this project (src/analysis.py)

# ---------------------------- CONFIGURATION ----------------------------
# None  -> look for the dataset inside the "data" folder automatically.
# Or give the full path, for example:  r"C:\Users\you\Downloads\Global_Superstore2.csv"
DATA_PATH = None
# -----------------------------------------------------------------------

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)       # charts are saved here

ra.set_style()                                   # one consistent chart style
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

FIGS = []      # list of charts saved by this notebook
INTERP = {}    # interpretation text for each chart (filled in Section 11)


def save_show(fig, filename, title):
    """Save a chart in outputs/figures and show it."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, bbox_inches="tight", dpi=150)
    FIGS.append({"file": filename, "title": title, "interpretation": INTERP.get(filename, "")})
    plt.show()


def interpret(filename):
    """Show the written interpretation of a chart (calculated from the data)."""
    display(Markdown("**Interpretation:** " + INTERP.get(filename, "This chart is not available for this dataset.")))


print("Python libraries ready. pandas", pd.__version__, "| numpy", np.__version__)
print("Project folder:", PROJECT_ROOT)

## 7. Data Loading

`ra.find_data_file()` finds the file, `ra.read_dataset()` reads it (it tries the right text encoding automatically), and `ra.standardise_columns()` renames columns such as `Sub-Category` to `sub_category` so that they are easy to type. `ra.check_schema()` checks that the essential columns exist and tells you clearly if something is missing.

We also keep a **snapshot** of the loaded data so that, at the end of cleaning, we can prove that the original data was never changed.

In [ ]:
data_file = ra.find_data_file(DATA_PATH)                       # stops with a clear message if the file is missing
df_loaded, load_info = ra.read_dataset(data_file)
df_raw, name_map = ra.standardise_columns(df_loaded)           # df_raw = the ORIGINAL data (never modified)
schema = ra.check_schema(df_raw)

df_snapshot = df_raw.copy(deep=True)                           # used later to prove the original is unchanged

print(f"File     : {load_info['file']} ({load_info['size_mb']} MB), text encoding used: {load_info['encoding']}")
print(f"Size     : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("Essential columns missing :", schema["missing_required"] or "none")
print("Optional columns not found:", schema["missing_optional"] or "none (all 24 documented columns present)")
print("Extra columns             :", schema["extra_columns"] or "none")

## 8. Initial Data Exploration

First we simply *look* at the data: its first and last rows, size, columns, data types and summary statistics.

### 8.1 First and last five rows

In [ ]:
df_raw.head()

In [ ]:
df_raw.tail()

### 8.2 Number of rows and columns, column names and data types

In [ ]:
print("Rows   :", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print()
print(list(df_raw.columns))
df_raw.dtypes.to_frame("data type")

**How to read this:** `object`/`str` means text, `int64` whole numbers, `float64` decimals. Dates are usually stored as *text* in a CSV file, so `order_date` and `ship_date` will appear as text and must be converted to real dates during cleaning.

### 8.3 Descriptive statistics

In [ ]:
# Numerical columns: count, mean, std, min, quartiles, max
df_raw.describe().T

In [ ]:
# Text columns: count, number of unique values, most frequent value (top) and its frequency
df_raw.select_dtypes(exclude="number").describe().T

### 8.4 Missing values (with percentages) and unique values

In [ ]:
missing = pd.DataFrame({"missing_count": df_raw.isna().sum(),
                        "missing_%": (df_raw.isna().mean() * 100).round(2)})
print("Columns with missing values:")
display(missing[missing["missing_count"] > 0] if (missing["missing_count"] > 0).any() else "No missing values at all")
print("Number of unique values in every column:")
df_raw.nunique().to_frame("unique values").T

### 8.5 Category counts (categorical variables)

In [ ]:
for col in ["category", "sub_category", "segment", "market", "region", "ship_mode", "order_priority"]:
    if col in df_raw.columns:
        print(f"--- {col} ({df_raw[col].nunique()} unique values) ---")
        print(df_raw[col].value_counts().head(12).to_string())
        print()

### 8.6 Numerical distributions (skewness) and date columns

In [ ]:
num_cols = [c for c in ["sales", "quantity", "discount", "profit", "shipping_cost"] if c in df_raw.columns]
numeric_view = df_raw[num_cols].apply(pd.to_numeric, errors="coerce")
print("Skewness (0 = symmetric, positive = long right tail, negative = long left tail):")
print(numeric_view.skew().round(2).to_string())
print()
for col in ["order_date", "ship_date"]:
    if col in df_raw.columns:
        parsed, fmt, failed = ra.detect_date_format(df_raw[col])
        print(f"{col}: example value '{df_raw[col].iloc[0]}', detected format {fmt}, values that cannot be parsed: {failed}")
        print(f"   range: {parsed.min().date()} to {parsed.max().date()}")

**How to read this:** a large positive skewness means most values are small and a few are very large. For such columns the **median** describes a typical value better than the mean. The date check shows the format used in the file (for example day-month-year) and the period covered.

## 9. Data Quality Assessment

Before cleaning, we measure the problems. Nothing is changed in this section.

* **Missing values** and **duplicates**
* **Invalid or impossible values** (negative sales, zero quantity, discount outside 0-1, shipped before ordered, profit larger than sales)
* **Outliers** with the IQR rule: a value is a potential outlier if it is below Q1 - 1.5xIQR or above Q3 + 1.5xIQR

In [ ]:
quality = ra.assess_quality(df_raw)

print("Rows:", f"{quality['rows']:,}", "| Columns:", quality["columns"])
print("Exact duplicate rows (identical in every column):", quality["exact_duplicates"])
print("Rows identical except Row ID:", quality["duplicates_ignoring_row_id"])
print("Row ID is unique:", quality["row_id_unique"])
print()
print("Missing values:")
display(quality["missing"] if len(quality["missing"]) else "none")
print("Invalid / impossible values (number of rows):")
quality["invalid_values"].to_frame("rows")

In [ ]:
# Outliers (IQR rule) for the main numerical columns
rows = []
for col in ["sales", "quantity", "discount", "profit", "shipping_cost"]:
    if col in df_raw.columns:
        s = pd.to_numeric(df_raw[col], errors="coerce")
        low, high = ra.iqr_bounds(s)
        n_out = int(((s < low) | (s > high)).sum())
        rows.append({"column": col, "lower_limit": low, "upper_limit": high, "outliers": n_out,
                     "outliers_%": n_out / s.notna().sum() * 100, "min": s.min(), "max": s.max()})
outlier_table = pd.DataFrame(rows).set_index("column")
outlier_table.round(2)

**Investigating unusual values before deciding anything.** Outliers are not automatically errors. We look at the most extreme rows to understand them.

In [ ]:
show_cols = [c for c in ["order_id", "order_date", "category", "sub_category", "product_name", "sales", "quantity", "discount", "profit"] if c in df_raw.columns]
view = df_raw.assign(profit=pd.to_numeric(df_raw["profit"], errors="coerce"), sales=pd.to_numeric(df_raw["sales"], errors="coerce"))

print("The 5 biggest losses (lowest profit):")
display(view.nsmallest(5, "profit")[show_cols])
print("The 5 biggest profits:")
display(view.nlargest(5, "profit")[show_cols])
print("The 5 biggest sales:")
display(view.nlargest(5, "sales")[show_cols])

In [ ]:
n_dup, n_invalid = quality["exact_duplicates"], int(quality["invalid_values"].drop(labels=["profit > sales", "discount > 1 (maybe stored as percent)"], errors="ignore").sum())
display(Markdown(f"""**Summary of the quality assessment (calculated from your file)**

* Exact duplicate rows: **{n_dup:,}**.
* Rows with impossible values (negative sales, zero quantity, negative discount or shipping cost, shipped before ordered): **{n_invalid:,}**.
* Columns with missing values: **{len(quality['missing'])}**{(' (' + ', '.join(quality['missing'].index) + ')') if len(quality['missing']) else ''}.
* Potential outliers by the IQR rule are listed in the table above.

**Decisions (made in Section 10):** remove exact duplicates and impossible records; fill small amounts of missing data and leave columns with many missing values unused; **keep outliers** because large orders and heavy losses are real business events, but flag them so they can be analysed separately."""))

## 10. Data Cleaning and Preprocessing

The original DataFrame `df_raw` is **never changed**. We copy it to `df` and apply seven steps. Each step is a function in `src/analysis.py` and writes a line to an **audit log** (rows before/after, rows affected, details), so nothing changes silently.

In [ ]:
audit = ra.AuditLog()          # records every step
df = df_raw.copy()             # all cleaning happens on this copy


def show_last(log):
    s = log.steps[-1]
    print(f"Step {s['step']}: {s['description']}")
    print(f"   rows: {s['rows_before']:,} -> {s['rows_after']:,}  (removed {s['rows_removed']:,}) | affected: {s['affected']}")
    print(f"   details: {s['details']}")

**Step 1: strip spaces from text columns.** Extra spaces make `'Toronto'` and `' Toronto'` look like two different cities. Empty text becomes a proper missing value.

In [ ]:
df = ra.step_strip_text(df, audit)
show_last(audit)

**Step 2: correct data types.** Dates become real `datetime` values (the format is detected automatically), money and quantity columns become numbers, and the discount scale is checked (it must be between 0 and 1).

In [ ]:
df = ra.step_convert_types(df, audit)
show_last(audit)
df[["order_date", "ship_date", "sales", "quantity", "discount", "profit"]].dtypes.to_frame("data type after conversion")

**Step 3: handle missing values.** Rows missing an essential value (order id, order date, sales, quantity, discount, profit) cannot be analysed and are dropped. For other columns: if less than 5% is missing we fill it (median for numbers, `'Unknown'` for text); if 5% or more is missing we do **not** invent data, we keep the column but do not use it.

In [ ]:
df = ra.step_handle_missing(df, audit)
show_last(audit)
print("Columns kept but not used because too many values are missing:", audit.notes.get("high_missing_columns") or "none")

**Step 4: remove exact duplicates.** Only rows that are identical in *every* column are removed. Rows that differ only in Row ID are reported but kept, because the same product can legitimately appear twice in an order.

In [ ]:
df = ra.step_remove_duplicates(df, audit)
show_last(audit)

**Step 5: remove impossible values.** Negative sales, zero or negative quantity, discount outside 0-1, negative shipping cost, or a ship date before the order date cannot be real. The number of affected rows is recorded.

In [ ]:
df = ra.step_remove_invalid(df, audit)
show_last(audit)

**Step 6: create new columns** that the analysis needs: year, month, quarter, shipping days, profit margin (profit / sales), discount band and a loss flag.

In [ ]:
df = ra.step_add_features(df, audit)
show_last(audit)
df[["order_date", "order_year", "order_month", "shipping_days", "profit_margin", "discount_band", "is_loss"]].head()

**Step 7: flag outliers (do not remove them).** The IQR rule marks outliers in new `outlier_...` columns. They stay in the data because they are genuine large orders or heavy losses, and removing valid observations would distort the totals.

In [ ]:
df = ra.step_flag_outliers(df, audit)
show_last(audit)
df = df.reset_index(drop=True)

### Cleaning audit, before/after comparison and validation

In [ ]:
display(audit.to_frame())

before_after = pd.DataFrame({
    "Before cleaning": [len(df_raw), int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum()), df_raw.shape[1]],
    "After cleaning":  [len(df), int(df.isna().sum().sum()), int(df.duplicated().sum()), df.shape[1]],
}, index=["Rows", "Missing cells", "Duplicate rows", "Columns"])
display(before_after)

# ---- validation checks (the notebook stops with an error if one of them fails) ----
pd.testing.assert_frame_equal(df_raw, df_snapshot)                       # original data is untouched
assert (df["sales"] >= 0).all() and (df["quantity"] > 0).all() and df["discount"].between(0, 1).all()
assert df[["sales", "profit", "quantity", "discount", "order_date"]].notna().all().all()
df_check, _ = ra.clean_data(df_raw)                                      # the one-call version gives the same result
pd.testing.assert_frame_equal(df, df_check)
print("All validation checks passed: original unchanged, no impossible values, step-by-step result equals ra.clean_data().")

## 11. Exploratory Data Analysis (EDA)

We now calculate all tables and key numbers with `ra.compute_results()`, then look at them one question at a time. `K` holds the key numbers (KPIs), `T` the tables and `S` the statistical test results.

In [ ]:
R = ra.compute_results(df)
K, T, S = R["kpis"], R["tables"], R["stats"]
INTERP.update(ra.figure_interpretations(R, df))      # chart interpretations, calculated from the data
print("Tables available:", ", ".join(T))

### 11.1 Total sales, total profit, orders and customers

In [ ]:
# The core numbers calculated directly with Pandas
print("Total sales        :", round(df["sales"].sum(), 2))
print("Total profit       :", round(df["profit"].sum(), 2))
print("Profit margin (%)  :", round(df["profit"].sum() / df["sales"].sum() * 100, 2))
print("Order lines        :", len(df))
print("Unique orders      :", df["order_id"].nunique())
if "customer_id" in df.columns:
    print("Unique customers   :", df["customer_id"].nunique())
print()
pd.Series(K, name="value").to_frame()

In [ ]:
display(Markdown(f"""**Interpretation:** between **{K['date_min']}** and **{K['date_max']}** the data contains **{K['n_orders']:,} orders** ({K['n_lines']:,} order lines) with total sales of **{K['total_sales']:,.0f}** and total profit of **{K['total_profit']:,.0f}**, a profit margin of **{K['profit_margin_pct']:.1f}%**. The average order is worth {K['avg_order_value']:,.0f} but the median order only {K['median_order_value']:,.0f}, so a few big orders pull the average up. **{K['loss_line_pct']:.1f}%** of order lines lose money (together {K['total_loss_amount']:,.0f})."""))

### 11.2 Sales and profit by category and sub-category

In [ ]:
display(T["category"][["sales", "profit", "profit_margin_pct", "quantity", "orders", "loss_line_pct"]])
T["sub_category"][["sales", "profit", "profit_margin_pct", "loss_line_pct"]]

In [ ]:
c = T["category"]
sc = T["sub_category"].reset_index()
loss_sc = sc[sc["profit"] < 0]
display(Markdown(f"""**Interpretation:** **{c['sales'].idxmax()}** has the highest sales and **{c['profit'].idxmax()}** has the highest profit; **{c['profit_margin_pct'].idxmin()}** has the lowest margin ({c['profit_margin_pct'].min():.1f}%). """ +
    (f"**{len(loss_sc)} sub-categories lose money overall:** {', '.join(loss_sc['sub_category'])}." if len(loss_sc) else "No sub-category has a negative total profit.")))

### 11.3 Sales and profit by market and region

In [ ]:
geo_tables = {k: T[k] for k in ["market", "region"] if k in T}
for name, tbl in geo_tables.items():
    print(f"--- by {name} ---")
    display(tbl[["sales", "profit", "profit_margin_pct", "orders", "loss_line_pct"]])
if "country" in T:
    print("--- top 15 countries by sales ---")
    display(T["country"][["sales", "profit", "profit_margin_pct"]])

In [ ]:
geo = "market" if "market" in T else "region"
g = T[geo]
display(Markdown(f"""**Interpretation:** by {geo}, **{g['sales'].idxmax()}** sells the most and **{g['profit'].idxmax()}** earns the most profit. The lowest margin is in **{g['profit_margin_pct'].idxmin()}** ({g['profit_margin_pct'].min():.1f}%): it sells a lot but keeps the least of each sale."""))

### 11.4 Yearly and monthly trends, and unusual periods

In [ ]:
display(T["year"][["sales", "profit", "sales_growth_pct", "profit_growth_pct", "profit_margin_pct"]])
monthly = T["monthly"]
print("Five strongest months:")
display(monthly.nlargest(5, "sales")[["sales", "profit", "z_score_sales"]])
print("Five weakest months:")
display(monthly.nsmallest(5, "sales")[["sales", "profit", "z_score_sales"]])
print("Average by calendar month (seasonality):")
T["seasonality"]

In [ ]:
seas = T["seasonality"]["sales"].sort_values(ascending=False)
unusual = monthly[monthly["z_score_sales"].abs() >= 2]
display(Markdown(f"""**Interpretation:** on average **{seas.index[0]}** and **{seas.index[1]}** are the strongest calendar months and **{seas.index[-1]}** the weakest. The z-score tells how many standard deviations a month is from the average month; """ +
    (f"months with |z| >= 2 are unusual: {', '.join(f'{d:%b %Y}' for d in unusual.index)}." if len(unusual) else "no month has |z| >= 2, so no month is statistically unusual.")))

### 11.5 Top-performing and underperforming products

In [ ]:
if "products_top_profit" in T:
    print("Top 10 products by profit:"); display(T["products_top_profit"])
    print("10 worst products by profit (largest total losses):"); display(T["products_bottom_profit"])
    print("Top 10 products by sales:"); display(T["products_top_sales"])

In [ ]:
if "n_products" in K:
    display(Markdown(f"""**Interpretation:** **{K['n_loss_products']:,} of {K['n_products']:,} products ({K['n_loss_products'] / K['n_products'] * 100:.1f}%)** have a negative total profit, losing {-K['loss_products_amount']:,.0f} in total. The single worst product is **{T['products_bottom_profit'].index[0]}**."""))

### 11.6 Customer purchasing patterns and segments

In [ ]:
if "customers_top" in T:
    print("Top 10 customers by sales:"); display(T["customers_top"])
if "segment" in T:
    print("By customer segment:"); display(T["segment"][["sales", "profit", "profit_margin_pct", "orders", "avg_order_value"]])

In [ ]:
if "repeat_customer_pct" in K:
    display(Markdown(f"""**Interpretation:** **{K['repeat_customer_pct']:.1f}%** of the {K['n_customers']:,} customers ordered more than once (average {K['avg_orders_per_customer']:.1f} orders each). The top 10 customers account for {K['top10_customers_sales_share_pct']:.1f}% of sales."""))

### 11.7 Discount versus profit

In [ ]:
T["discount_band"][["lines", "sales", "profit", "profit_margin_pct", "median_margin", "loss_line_pct"]]

In [ ]:
d = T["discount_band"]
sp = S.get("spearman_discount_margin")
display(Markdown(f"""**Interpretation:** the profit margin is **{d['profit_margin_pct'].iloc[0]:.1f}%** for order lines without discount and **{d['profit_margin_pct'].iloc[-1]:.1f}%** in the highest discount band; the share of loss-making lines rises from {d['loss_line_pct'].iloc[0]:.0f}% to {d['loss_line_pct'].iloc[-1]:.0f}%. """ +
    (f"The Spearman correlation between discount and margin is **{sp['rho']:.2f}**. " if sp else "") +
    "This is an association in observed data; it does not prove that discounts alone *cause* the losses."))

### 11.8 Loss-making products and categories

In [ ]:
lines_loss = df.groupby("category", observed=True)["is_loss"].agg(["sum", "mean"]).rename(columns={"sum": "loss_lines", "mean": "loss_share"}) if "category" in df.columns else None
print("Loss-making order lines by category:")
display(lines_loss)
print("Sub-categories with a negative total profit:")
display(T["sub_category"][T["sub_category"]["profit"] < 0][["sales", "profit", "profit_margin_pct"]] if "sub_category" in T else "sub_category not available")

### 11.9 Shipping

In [ ]:
if "ship_mode" in T:
    display(T["ship_mode"][["lines", "sales", "profit", "profit_margin_pct"]])
if "shipping_days" in df.columns:
    print("Average shipping days by ship mode:")
    display(df.groupby("ship_mode")["shipping_days"].agg(["mean", "median", "max"]) if "ship_mode" in df.columns else df["shipping_days"].describe())

## 12. Statistical Analysis

### 12.1 Descriptive statistics, skewness and kurtosis
Skewness measures asymmetry (0 = symmetric); kurtosis measures how heavy the tails are (0 = like a normal distribution).

In [ ]:
T["describe"][["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew", "kurtosis"]]

In [ ]:
desc = T["describe"]
display(Markdown(f"""**Interpretation:** sales has a skewness of **{desc.loc['sales', 'skew']:.1f}** and profit **{desc.loc['profit', 'skew']:.1f}**, and the kurtosis values are far from 0. The data is **not normally distributed**: it has extreme values. Therefore we use the **median** and **rank-based (non-parametric) tests** (Spearman, Kruskal-Wallis, Mann-Whitney) instead of tests that assume a normal distribution."""))

### 12.2 Correlation (Pearson and Spearman)
Pearson measures *straight-line* relationships and is sensitive to outliers; Spearman works on ranks and is more robust. Values go from -1 to +1. **Correlation does not prove causation.**

In [ ]:
print("Pearson correlation:"); display(T["corr_pearson"].round(2))
print("Spearman correlation:"); T["corr_spearman"].round(2)

In [ ]:
def strength(r):
    r = abs(r)
    return "negligible" if r < 0.1 else "weak" if r < 0.3 else "moderate" if r < 0.5 else "strong"

pairs = [("discount", "profit"), ("discount", "profit_margin"), ("sales", "profit"), ("quantity", "sales")]
lines = []
for a, b in pairs:
    if a in T["corr_spearman"].index and b in T["corr_spearman"].columns:
        r = T["corr_spearman"].loc[a, b]
        lines.append(f"* **{a} vs {b}:** Spearman = {r:.2f} ({strength(r)}, {'positive' if r > 0 else 'negative'})")
display(Markdown("**Interpretation (Spearman):**\n\n" + "\n".join(lines)))

### 12.3 Statistical tests on discounts
* **Spearman test:** is there a monotonic relationship between discount and profit margin?
* **Kruskal-Wallis test:** do the profit margins differ between the discount bands?
* **Mann-Whitney U test:** do discounted and non-discounted order lines have different margins?

With tens of thousands of rows, p-values become tiny even for small effects, so we always look at the **size of the effect** (for example the difference in medians) and not only at the p-value.

In [ ]:
rows = [{"test": name, **vals} for name, vals in S.items() if name != "outlier_sensitivity"]
pd.DataFrame(rows).set_index("test") if rows else "Not enough data for the tests"

In [ ]:
mw = S.get("mannwhitney_discount_vs_none")
kw = S.get("kruskal_margin_by_discount_band")
text = []
if kw:
    text.append(f"* Kruskal-Wallis: H = {kw['H']:.1f}, p = {kw['p_value']:.2g} -> " + ("the profit margin **differs between discount bands** (p < 0.05)." if kw["p_value"] < 0.05 else "no evidence of a difference between bands (p >= 0.05)."))
if mw:
    text.append(f"* Mann-Whitney: median margin is **{mw['median_margin_no_discount'] * 100:.1f}%** without discount and **{mw['median_margin_discounted'] * 100:.1f}%** with discount (p = {mw['p_value']:.2g}). The difference in medians ({(mw['median_margin_no_discount'] - mw['median_margin_discounted']) * 100:.1f} percentage points) is the practical effect size.")
display(Markdown("**Interpretation:**\n\n" + "\n".join(text) + "\n\nA small p-value means the difference is unlikely to be random chance; it does **not** prove that discounts are the cause."))

### 12.4 How much do outliers influence the results?

In [ ]:
pd.Series(S["outlier_sensitivity"], name="value").to_frame()

## 13. Data Visualizations

Every chart has a title, labelled axes, readable labels and a legend when needed, and is saved as a PNG file in `outputs/figures`. The chart-drawing functions are in `src/analysis.py` (each one is short and uses Matplotlib / Seaborn); the interpretation below each chart is written from the calculated results.

### 13.1 Bar chart: sales and profit by category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ra.plot_category_bars(ax, T["category"], "Category")
ax.set_title("Total Sales and Profit by Category")
save_show(fig, "01_sales_profit_by_category.png", "Total Sales and Profit by Category")
interpret("01_sales_profit_by_category.png")

### 13.2 Bar chart: profit by sub-category (losses in red)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ra.plot_subcategory_profit(ax, T["sub_category"])
ax.set_title("Total Profit by Sub-Category (red = loss)")
save_show(fig, "02_profit_by_subcategory.png", "Total Profit by Sub-Category")
interpret("02_profit_by_subcategory.png")

### 13.3 Bar chart: sales and profit by market

In [ ]:
geo = "market" if "market" in T else "region"
fig, ax = plt.subplots(figsize=(9, 5))
ra.plot_category_bars(ax, T[geo], geo.title())
ax.set_title(f"Total Sales and Profit by {geo.title()}")
save_show(fig, "03_sales_profit_by_market.png", f"Total Sales and Profit by {geo.title()}")
interpret("03_sales_profit_by_market.png")

### 13.4 Heatmap: profit margin by market and category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ra.plot_margin_heatmap(ax, df, geo, "category")
ax.set_title(f"Profit Margin (%) by {geo.title()} and Category")
save_show(fig, "04_margin_heatmap_market_category.png", f"Profit Margin by {geo.title()} and Category")
interpret("04_margin_heatmap_market_category.png")

### 13.5 Histograms: distribution of sales and profit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ra.plot_distributions(axes, df)
save_show(fig, "05_distributions.png", "Distributions of Sales and Profit")
interpret("05_distributions.png")

### 13.6 Line chart: monthly sales and profit trend

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ra.plot_monthly_trend(ax, T["monthly"])
ax.set_title("Monthly Sales and Profit Trend")
save_show(fig, "06_monthly_trend.png", "Monthly Sales and Profit Trend")
interpret("06_monthly_trend.png")

### 13.7 Line chart: yearly sales by category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ra.plot_yearly_category(ax, df)
ax.set_title("Yearly Sales by Category")
save_show(fig, "07_yearly_sales_by_category.png", "Yearly Sales by Category")
interpret("07_yearly_sales_by_category.png")

### 13.8 Scatter plot and bars: discount versus profit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ra.plot_discount(axes, df, T["discount_band"])
save_show(fig, "08_discount_vs_profit.png", "Discount versus Profit")
interpret("08_discount_vs_profit.png")

### 13.9 Box plot: profit per order line by category (outlier analysis)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ra.plot_box_profit(ax, df)
ax.set_title("Profit per Order Line by Category (outliers shown)")
save_show(fig, "09_profit_boxplot_by_category.png", "Profit per Order Line by Category")
interpret("09_profit_boxplot_by_category.png")

### 13.10 Heatmap: correlation matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
ra.plot_corr(ax, T["corr_pearson"])
ax.set_title("Correlation Heatmap of Numerical Variables")
save_show(fig, "10_correlation_heatmap.png", "Correlation Heatmap")
interpret("10_correlation_heatmap.png")

### 13.11 Top and bottom products

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ra.plot_top_bottom_products(axes, T["products_top_profit"], T["products_bottom_profit"])
save_show(fig, "11_top_bottom_products.png", "Top and Bottom Products by Profit")
interpret("11_top_bottom_products.png")

### 13.12 Shipping analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ra.plot_shipping(axes, df, T["ship_mode"])
save_show(fig, "12_shipping_analysis.png", "Shipping Analysis")
interpret("12_shipping_analysis.png")

### 13.13 Heatmap: sales by year and month (seasonality)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ra.plot_seasonality_heatmap(ax, T["sales_year_month"])
ax.set_title("Sales by Year and Month (thousands)")
save_show(fig, "13_sales_heatmap_year_month.png", "Sales Heatmap by Year and Month")
interpret("13_sales_heatmap_year_month.png")

## 14. Key Findings and Business Insights

Each finding is generated from the calculated results and has four parts: **observation**, **numerical evidence**, **business significance** and **recommendation**. They are drafted automatically so that no number is invented; **read them, check them against the tables above and rewrite them in your own words** in your report.

In [ ]:
insights = ra.build_insights(R)
for i, ins in enumerate(insights, 1):
    display(Markdown(f"""**{i}. {ins['title']}**

* **Observation:** {ins['observation']}
* **Evidence:** {ins['evidence']}
* **Business significance:** {ins['significance']}
* **Recommendation:** {ins['recommendation']}"""))

## 15. Recommendations

In [ ]:
display(Markdown("**Actionable recommendations based on the findings above:**\n\n" +
                 "\n".join(f"{i}. **{ins['title']}:** {ins['recommendation']}" for i, ins in enumerate(insights, 1))))

## 16. Limitations of the Analysis

* **Association, not causation.** Correlations and group differences describe the data; they do not prove that one factor causes another (for example, deep discounts coincide with low margins, but product mix and costs also differ).
* **One dataset.** Results describe this file only and may not hold for other periods, companies or countries.
* **Unknown definitions.** The meaning of `Profit` (which costs are included) and the currency are not documented in the sources I could verify.
* **Outliers kept.** Extreme orders influence totals and means; medians and non-parametric tests were used to reduce this.
* **Missing postal codes / unused columns.** Columns with many missing values were not used (see the cleaning audit).
* **No forecasting or modelling.** The project is descriptive. Machine learning and forecasting are future work.
* **Automatically drafted insights.** The text in Section 14 is generated by rules from the results and must be reviewed by a person before it is submitted.

In [ ]:
display(Markdown(f"""## 17. Conclusion

This project loaded and cleaned the Global Superstore data ({len(df_raw):,} raw rows, {len(df):,} rows after cleaning), documented every cleaning step, and analysed {K['n_orders']:,} orders from {K['date_min']} to {K['date_max']}. Total sales were **{K['total_sales']:,.0f}** and total profit **{K['total_profit']:,.0f}** (margin **{K['profit_margin_pct']:.1f}%**). The analysis showed which categories, markets and products drive profit, where money is lost ({K['loss_line_pct']:.1f}% of order lines are loss-making), how sales change over time, and how profit margin changes with discount level. Pandas and NumPy handled the data, SciPy supported the statistical tests, and Matplotlib and Seaborn turned the results into clear charts.

*(Edit this paragraph in your own words before submission and add what you personally learned.)*"""))

## 18. References

1. Kaggle dataset: *Global Superstore*, `apoorvaappz/global-super-store-dataset`. https://www.kaggle.com/datasets/apoorvaappz/global-super-store-dataset
2. The pandas development team. *pandas documentation.* https://pandas.pydata.org/docs/
3. NumPy Developers. *NumPy documentation.* https://numpy.org/doc/
4. SciPy Developers. *SciPy documentation (scipy.stats).* https://docs.scipy.org/doc/scipy/
5. Matplotlib Development Team. *Matplotlib documentation.* https://matplotlib.org/stable/
6. Waskom, M. L. *seaborn: statistical data visualization.* https://seaborn.pydata.org/
7. Project Jupyter. *Jupyter Notebook documentation.* https://jupyter-notebook.readthedocs.io/

In [ ]:
# Save the real results (tables, summary.json) so the report builder can use them
ra.save_outputs(OUTPUT_DIR, load_info, schema, quality, audit, R, insights, FIGS, len(df_raw), len(df))
print("Saved in", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file() and p.name != ".gitkeep":
        print(" -", p.relative_to(OUTPUT_DIR))